# Optimization: Driving a Case Toward a Design Target

Notebook 01 ran a fixed design of experiments: every set of parameters was decided
up front, and the samples were independent of one another. Here the next set of
parameters depends on what the previous run returned, because an optimizer is
choosing them.

The case is the same passive scalar transport through the `pitzDaily` geometry used
in notebook 01, with the same single parameter, the diffusivity `DT`. One scalar
unknown on a cheap case keeps the whole search to a couple of dozen simulations,
which is small enough to watch it run.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

import uqtopus as uqt

import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

## 1. The simulator as a callable

`OpenFOAMSimulator` puts the template and the solver script behind a single
`run(params)` call that returns the parsed fields as an `xr.Dataset`. Each call is
one full simulation, so `run_count` doubles as the budget meter for everything
below.

`qoi_variables` and `qoi_times` narrow what gets read back from disk after each
run. Here only the scalar `T` at the final time is needed, so there is no reason to
parse the velocity field or the twenty intermediate write times.

In [ ]:
PARAM_KEY = 'constant__transportProperties__DT'

sim = uqt.OpenFOAMSimulator(
    template_path="templates/scalarTransportFoam",
    solver_script="Allrun",
    output_path="experiments/optimization",
    qoi_variables=["T"],
    qoi_times=["0.1"]
)
sim

## 2. The quantity of interest

`DT` sets how fast the tracer entering at the inlet diffuses into the rest of the
domain. Averaging `T` over every cell at the end time turns that into one number:
the more the scalar has spread, the higher the mean.

Any function of the returned dataset works here. This one is deliberately trivial,
so that the notebook is about the optimization loop rather than about the QoI.

In [ ]:
def mean_T(ds):
    """Domain-averaged scalar at the final stored time."""
    return float(ds['T'].values[-1].mean())

## 3. Bracket the design space

Before optimizing, it is worth knowing which targets are reachable at all. Two runs
at the ends of the parameter range give the span of achievable QoI values, and show
that the QoI moves monotonically with `DT`.

Skipping this step is the usual way to end up with an optimizer grinding through
dozens of simulations against a target that no parameter value could ever produce.

In [ ]:
DT_LOW, DT_HIGH = 0.01, 0.3

qoi_low  = mean_T(sim.run({PARAM_KEY: DT_LOW}))
qoi_high = mean_T(sim.run({PARAM_KEY: DT_HIGH}))

print(f"DT = {DT_LOW:<5} -> mean T = {qoi_low:.4f}")
print(f"DT = {DT_HIGH:<5} -> mean T = {qoi_high:.4f}")
print(f"\nreachable range: [{min(qoi_low, qoi_high):.4f}, {max(qoi_low, qoi_high):.4f}]")

## 4. Optimize

The design requirement is a target mean scalar, picked inside the bracket above so
it is known to be attainable. Squaring the mismatch turns "hit this value" into
something to minimize.

`as_objective` is the part that matters: it wraps the simulator into a plain
callable taking an array of parameter values and returning a float. SciPy never
learns that OpenFOAM is on the other end, and the optimization loop stays entirely
inside SciPy. The callable expects one entry per name in `param_keys`, so the scalar
that `minimize_scalar` proposes is wrapped in a list.

In [ ]:
TARGET = 0.5 * (qoi_low + qoi_high)   # midway between the two bracket runs

qoi_history, dt_history = [], []

def objective(ds):
    qoi = mean_T(ds)
    qoi_history.append(qoi)
    return (qoi - TARGET) ** 2

f = sim.as_objective(metric_fn=objective, param_keys=[PARAM_KEY])

def tracked(dt):
    value = f([dt])          # one entry per name in param_keys
    dt_history.append(dt)
    return value

sim.reset()
result = minimize_scalar(tracked, bounds=(DT_LOW, DT_HIGH), method='bounded')

DT_opt = float(result.x)
best = int(np.argmin([(q - TARGET) ** 2 for q in qoi_history]))

print(f"target mean T   {TARGET:.4f}")
print(f"achieved        {qoi_history[best]:.4f}")
print(f"optimal DT      {DT_opt:.4f}")
print(f"simulations     {sim.run_count}")

## 5. The search

Every point below cost one OpenFOAM run. The bounded Brent method spends its first
evaluations spanning the interval and then clusters them around the solution, which
is what the crowding near the target line shows.

In [ ]:
order = np.argsort(dt_history)
dt_sorted = np.array(dt_history)[order]
qoi_sorted = np.array(qoi_history)[order]

fig, ax = plt.subplots(figsize=(5, 3), constrained_layout=True)

ax.axhline(TARGET, color='k', linestyle='--', linewidth=1, label='target')
ax.plot(dt_sorted, qoi_sorted, 'o-', color='tab:gray', markersize=4,
        linewidth=0.8, label='evaluations')
ax.plot(DT_opt, TARGET, '*', color='k', markersize=14, label='optimum')

ax.set_xlabel('DT')
ax.set_ylabel('mean T')
ax.legend(frameon=False, fontsize=9)

plt.show()

Two things are worth carrying over to a case that is not this cheap. Every marker on
that plot is a full solver run, so the cost of the study is set by how many
evaluations the optimizer needs, not by how the objective is written. And each run
is kept on disk under `experiments/optimization`, one directory per evaluation, so a
long search over a large case is also a disk-space decision.

Notebook 05 keeps the same machinery but points it at measured data instead of a
design target, and fits two parameters at once.